# BSL в Jupyter: от первой ячейки до живого проведения ЗУП

Этот notebook — обзорный сценарий 1C Interactive Runtime. Начнём с обычного BSL-кода, сохраним состояние между ячейками, вызовем типовые методы настоящей ЗУП, перенесём результат в Python, а затем коротко покажем две более глубокие возможности — hot reload и capture внутри проведения документа.

Hot reload и capture здесь показаны намеренно без разбора внутреннего устройства: каждому механизму посвящена отдельная статья.

**Маршрут опыта:**

`BSL → persistent state → код ЗУП → pandas → hot reload → capture → resume`

Конфигурация: ЗУП КОРП 3.1.38.92; дата снимка данных: 01.08.2021. Используйте отдельную демо-копию и соответствующую ей выгрузку исходников. Подготовка окружения — в [README](../README.md). Сеанс закрывается последней ячейкой; при прерывании выполните `runtime.close()`.

## Подготовка

Укажите `PLATFORM_BIN`, `CONNECTION_STRING` и `SOURCE_ROOT` для отдельной копии ЗУП и выгрузки её исходников. Если нужны учётные данные, задайте их локально в `RuntimeConfig`.

Эта служебная ячейка открывает один сеанс 1С. Все следующие BSL-ячейки работают в нём же и видят общий notebook-контекст.

In [ ]:
from IPython.display import display
from onec_runtime.config import RuntimeConfig
from onec_runtime.session import ExtensionMode, RuntimeSessionConfig
from onec_runtime_jupyter import InteractiveRuntimeSession

PLATFORM_BIN = r'C:\path\to\1cv8\bin'
CONNECTION_STRING = r'File="C:\path\to\ZUP-demo-copy";'
SOURCE_ROOT = r'C:\path\to\ZUP-source'
EXTENSION_MODE = ExtensionMode.AUTO  # Совместимое расширение проверяется при запуске.

runtime = InteractiveRuntimeSession.start(
    RuntimeSessionConfig(
        runtime=RuntimeConfig(
            platform_bin=PLATFORM_BIN,
            connection_string=CONNECTION_STRING,
            # При необходимости задайте username и password локально.
        ),
        source_root=SOURCE_ROOT,
        extension_mode=EXTENSION_MODE,
    )
)

## Первая BSL-ячейка

`%%bsl` выполняет содержимое ячейки как BSL в настоящем сеансе 1С. Это не интерпретатор BSL в Python: код выполняется платформой 1С, а результат возвращается в notebook.

In [ ]:
%%bsl
Сообщить("Привет, мир!");

## Состояние между ячейками

Notebook хранит BSL-контекст между выполнениями. Сохраним значение в одной ячейке и используем его в следующей.

In [ ]:
%%bsl
ПроцентПовышения = 10;

In [ ]:
%%bsl
Сообщить(ПроцентПовышения);

## В контексте остаются и методы

Состояние — это не только значения переменных. Объявленная функция также остаётся доступной в следующих BSL-ячейках и читает актуальное значение `ПроцентПовышения`.

In [ ]:
%%bsl
Функция УвеличитьНаПроцент(Значение)
    Возврат Значение * (1 + ПроцентПовышения / 100);
КонецФункции

In [ ]:
%%bsl
Сообщить(УвеличитьНаПроцент(100000));

In [ ]:
%%bsl
ПроцентПовышения = 20;

In [ ]:
%%bsl
Сообщить(УвеличитьНаПроцент(100000));

Первый вызов выводит 110000, второй — 120000. Функцию мы не переопределяли: изменилось только значение `ПроцентПовышения` в общем notebook-контексте.

На этом простом примере видна ключевая модель работы runtime: последовательность ячеек образует один интерактивный эксперимент, а не набор независимых запусков.

## Та же сессия — внутри настоящей ЗУП

Теперь вместо учебного примера вызовем типовой код конфигурации. `КадровыйУчет.СотрудникиОрганизации` выполняется в той же сессии 1С и возвращает реальные данные демобазы.

Дата 01.08.2021 относится к данным демоснимка, а 3.1.38.92 — к версии конфигурации.

In [ ]:
%%bsl
ДатаФОТ = Дата(2021, 8, 1);
ПараметрыСотрудников = КадровыйУчет.ПараметрыПолученияСотрудниковОрганизацийПоСпискуФизическихЛиц();
ПараметрыСотрудников.НачалоПериода = ДатаФОТ;
ПараметрыСотрудников.ОкончаниеПериода = ДатаФОТ;
ПараметрыСотрудников.РаботникиПоТрудовымДоговорам = Истина;
ПараметрыСотрудников.КадровыеДанные = "Организация,Подразделение,Должность,ФОТ";
КадровыеДанные = КадровыйУчет.СотрудникиОрганизации(
    Истина, ПараметрыСотрудников);

## Из 1С в Python

In [ ]:
# КадровыеДанные пока живёт в 1С; to_df() переносит таблицу в pandas.
df = КадровыеДанные.to_df(refs="presentation")
display(df.head(10))
print("Строк с кадровыми данными:", len(df))

Для ссылочных значений можно выбрать представление, UUID или оба значения. В обзорном сценарии представления удобны для чтения, а UUID пригодятся, когда нужно устойчиво сравнивать и соединять данные.

In [ ]:
df_uuid = КадровыеДанные.to_df(refs="uuid")
df_both = КадровыеДанные.to_df(refs="both")
display(df_uuid[["Сотрудник", "Подразделение"]].head())
display(df_both[["Сотрудник", "Сотрудник__uuid", "Подразделение"]].head())

## 1С получает данные, Python исследует их

Сгруппируем ФОТ по подразделению и построим диаграмму обычными средствами pandas/matplotlib. Важна граница ответственности: данные и расчёт пришли из ЗУП, Python используется как интерактивный аналитический слой.

In [ ]:
import matplotlib.pyplot as plt

by_department = (
    df.groupby("Подразделение", dropna=False)["ФОТ"]
      .sum()
      .sort_values()
)
fig, ax = plt.subplots(figsize=(9, max(4, len(by_department) * 0.38)))
by_department.map(float).plot.barh(ax=ax)
ax.set_xlabel("ФОТ, ₽")
ax.set_title("Плановый ФОТ по подразделениям на 01.08.2021")
fig.tight_layout()
display(fig)
plt.close(fig)

## Зафиксируем результат до экспериментов

Перед следующими опытами сохраним плановый ФОТ, который возвращает типовой метод `ТекущиеДанныеОплатыТрудаСотрудников`. Позже сможем сравнить результат с изменённым поведением.

In [ ]:
%%bsl
ПлановыйФот = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, КадровыеДанные);

In [ ]:
display(ПлановыйФот.head(10).to_df(refs="presentation"))

## Hot reload: короткий опыт

До сих пор мы не меняли код конфигурации. Теперь проверим другую гипотезу: локально изменим один типовой метод и загрузим его реализацию в текущий экспериментальный сеанс **без обновления конфигурации ИБ**.

В **локальной копии** выгрузки откройте `CommonModules/ПлановыеНачисленияСотрудников/Ext/Module.bsl`. В запросе метода `ТекущиеДанныеОплатыТрудаСотрудников` добавьте поле:

```bsl
ЗначенияСовокупныхТарифныхСтавок.ФОТ * 1.30 КАК ФОТСоСтраховыми
```

Исходный `ФОТ` остаётся в результате; новое поле показывает условную надбавку 30%.

Загрузим изменённый модуль в текущий сеанс и повторим тот же вызов. Механизм hot reload, его границы и сценарии использования подробно разбираются в отдельной статье.

In [ ]:
RELOAD_MODULE_PATH = r'CommonModules\ПлановыеНачисленияСотрудников\Ext\Module.bsl'
runtime.load_worker_module(RELOAD_MODULE_PATH)

In [ ]:
%%bsl
ПлановыйФотСоСтраховыми = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, КадровыеДанные);

In [ ]:
display(ПлановыйФотСоСтраховыми.to_df(refs="presentation"))

Изменённый метод нужен здесь только как демонстрация возможности. Теперь перейдём ко второй границе runtime: данным, которые существуют лишь внутри уже выполняющегося типового кода.

## Capture: остановка внутри проведения «Приема на работу»

Повторное проведение документа остановим внутри `РасчетЗарплатыРасширенный` перед обработкой таблиц плановых начислений и значений показателей.

В отличие от обычного запроса к уже записанным данным, capture даёт доступ к состоянию **текущего вызова**, пока он ещё не завершён. В этом обзорном notebook мы покажем только основной цикл: остановить → посмотреть → изменить → продолжить. Подробная механика capture разбирается отдельно.

Для финального опыта возвращаем повышение к 10%.

In [ ]:
%%bsl
ПроцентПовышения = 10;

In [ ]:
%%bsl
ЗапросПриемов = Новый Запрос;
ЗапросПриемов.Текст =
    "ВЫБРАТЬ
    |   Прием.Ссылка КАК Ссылка
    |ИЗ
    |   Документ.ПриемНаРаботу КАК Прием
    |ГДЕ
    |   Прием.Номер = &Номер";
ЗапросПриемов.УстановитьПараметр("Номер", "0000-000020");
Выборка =  ЗапросПриемов.Выполнить().Выбрать();
Если Выборка.Следующий() Тогда
    Прием = Выборка.Ссылка.ПолучитьОбъект();
    Сообщить("Прием с номером 0000-000020 найден: " + Прием.Дата+ ", " + Прием.Организация.Наименование);
Иначе
    Сообщить("Прием с номером 0000-000020 не найден");
КонецЕсли;
// Зафиксировать исходные значения для проверок после повторного проведения.
СсылкаНаПрием = Прием.Ссылка;
СотрудникДляОпыта = Прием.Сотрудник;
ОкладДляОпытаНайден = Истина;
ДокументОбъект = СсылкаНаПрием.ПолучитьОбъект();

In [ ]:
from pathlib import Path

CAPTURE_MODULE_PATH = r'CommonModules\РасчетЗарплатыРасширенный\Ext\Module.bsl'

runtime.add_capture_point(CAPTURE_MODULE_PATH, 768)

In [ ]:
%%bsl
Прием.Записать(РежимЗаписиДокумента.Проведение);

In [ ]:
capture_status = runtime.status()
capture_status.state.value

In [ ]:
runtime.current_capture().stack[:3].with_methods()

In [ ]:
%%bsl
Процедура ПоказатьОклад(ДокументыСсылка)
    ЗапросПроверки = Новый Запрос;
    ЗапросПроверки.Текст =
        "ВЫБРАТЬ ПЕРВЫЕ 1
        |   Значения.Показатель КАК Показатель,
        |   Значения.Значение КАК Значение
        |ИЗ
        |   РегистрСведений.ЗначенияПериодическихПоказателейРасчетаЗарплатыСотрудников КАК Значения
        |ГДЕ
        |   Значения.Регистратор = &ДокументыСсылка
        |   И Значения.Показатель.Наименование  = &ПоказательНаименование
        |УПОРЯДОЧИТЬ ПО
        |   Значения.Период УБЫВ";
    ЗапросПроверки.УстановитьПараметр("ДокументыСсылка", ДокументыСсылка);
    ЗапросПроверки.УстановитьПараметр("ПоказательНаименование", "Оклад");
    Выборка = ЗапросПроверки.Выполнить().Выбрать();
    Если Выборка.Следующий() Тогда
        Сообщить("Оклад: " + Выборка.Значение);
    Иначе
        Сообщить("Оклад не найден");
    КонецЕсли;
КонецПроцедуры
ПоказатьОклад(Прием.Ссылка);

Проведение ещё не завершилось. Runtime удерживает тот же вызов, а notebook может исследовать его текущий контекст.

In [ ]:
%%bsl
Сообщить(Строка(КонтекстОтладки.СтруктураДанных.ЗначенияПоказателей[0].Показатель) +
    " :" + КонтекстОтладки.СтруктураДанных.ЗначенияПоказателей[0].Значение);

In [ ]:
%%bsl
Процедура УвеличитьОклад(СтрокаПоказателя)
    СтрокаПоказателя.Значение = УвеличитьНаПроцент(СтрокаПоказателя.Значение);
КонецПроцедуры

In [ ]:
%%bsl
УвеличитьОклад(КонтекстОтладки.СтруктураДанных.ЗначенияПоказателей[0]);

Сообщить(Строка(КонтекстОтладки.СтруктураДанных.ЗначенияПоказателей[0].Показатель) +
    " :" + КонтекстОтладки.СтруктураДанных.ЗначенияПоказателей[0].Значение);

In [ ]:
completed = runtime.resume_capture()
runtime.clear_capture_points()
print('Продолжилось и завершилось то же проведение')

## Продолжаем тот же вызов и проверяем результат

После изменения данных снимаем capture и продолжаем **то же самое проведение**, а не запускаем документ заново. Затем читаем записанный результат и сравниваем его с исходным состоянием.

In [ ]:
%%bsl
ПоказатьОклад(Прием.Ссылка);

Типовой метод ниже показывает плановый ФОТ на дату снимка 01.08.2021. Если после выбранного приема были другие кадровые события, повторное проведение раннего документа не обязано изменить актуальный срез на эту дату. Поэтому основной факт эксперимента — изменение данных внутри capture и успешное завершение того же проведения.

In [ ]:
%%bsl
ПланПослеПроведения = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, КадровыеДанные)
    .Скопировать(Новый Структура("Сотрудник", СотрудникДляОпыта), "Сотрудник,ФОТ, СовокупнаяТарифнаяСтавка, ВидТарифнойСтавки");

In [ ]:
ПланПослеПроведения.to_df()

In [ ]:
runtime.close()